# Annual Balance Dashboard

Notebook gerencial para leer el backend anual de accounting.

**Diseño:** filas = líneas contables/gerenciales, columnas = años, y `Currency` como columna al lado de la métrica.  
**Regla fuerte:** ARS y USD nunca se suman. Si una métrica existe en ambas monedas, se muestran dos filas.  
**Rol de la notebook:** consumir métricas y contratos ya producidos por el backend; no corregir ledger, no reclasificar flows, no inferir caja.

Este dashboard está pensado como un paquete de estados gerenciales:

1. Estado de resultado operativo.
2. Puente de fondos y distribuciones.
3. Canales de pago / settlement.
4. Estado de posición operativa.
5. Movimiento de deuda interna.
6. Actor netting / claims.
7. Calidad de datos y salvedades.
8. Inventario de métricas y appendix.

## 0. Setup y carga del bundle latest

La notebook asume que se corre desde el root del repo o desde algún subdirectorio del repo. Busca automáticamente:

```text
public/accounting/latest
```

La fuente principal es:

```text
public/accounting/latest/canonical_dashboard/annual_balance_dashboard_metrics.csv
```

In [1]:
def normalize_dashboard_metrics(df):
    df = df.copy()

    # Normalize column names lightly
    df.columns = [str(c).strip() for c in df.columns]

    # Detect value column
    value_candidates = [
        "value",
        "metric_value",
        "amount",
        "annual_value",
        "close_amount",
        "net_amount",
    ]
    found_value_cols = [c for c in value_candidates if c in df.columns]

    if not found_value_cols:
        raise ValueError(
            "Could not find a numeric value column. Available columns: "
            + ", ".join(df.columns)
        )

    value_col = found_value_cols[0]

    if value_col != "value":
        df["value"] = df[value_col]

    df["value"] = pd.to_numeric(df["value"], errors="coerce")

    # Normalize period/year
    if "period" not in df.columns:
        raise ValueError("Expected a period column in annual dashboard metrics.")

    df["period"] = (
        df["period"]
        .astype(str)
        .str.strip()
        .str.replace(r"\.0$", "", regex=True)
    )

    # Normalize Currency
    if "Currency" not in df.columns:
        df["Currency"] = "N/A"

    df["Currency"] = df["Currency"].fillna("N/A").astype(str).str.strip()

    # Normalize common optional columns
    for c in [
        "metric_id",
        "dimension_name",
        "dimension_value",
        "dashboard_section",
        "value_status",
        "caveat",
        "source_table",
    ]:
        if c not in df.columns:
            df[c] = ""

    return df


def inspect_metric_values(df, metric_ids=None, top=80):
    x = df.copy()

    if metric_ids is not None:
        x = x[x["metric_id"].isin(metric_ids)]

    cols = [
        "metric_id",
        "Currency",
        "period",
        "dimension_name",
        "dimension_value",
        "value",
        "value_status",
        "source_table",
        "caveat",
    ]
    cols = [c for c in cols if c in x.columns]

    out = (
        x[cols]
        .sort_values(["metric_id", "Currency", "dimension_name", "dimension_value", "period"])
        .head(top)
    )

    print("rows:", len(x))
    print("non-null values:", x["value"].notna().sum())
    print("null values:", x["value"].isna().sum())

    display(out)


In [2]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
from IPython.display import display, Markdown

pd.set_option("display.max_columns", 200)
pd.set_option("display.max_rows", 200)
pd.set_option("display.width", 240)


def find_repo_root(start=None):
    start = Path(start or Path.cwd()).resolve()
    for p in [start, *start.parents]:
        if (p / "Makefile").exists() and (p / "public" / "accounting").exists():
            return p
    return start


REPO_ROOT = find_repo_root()
PUBLIC_BASE = REPO_ROOT / "public" / "accounting" / "latest"
DASH_DIR = PUBLIC_BASE / "canonical_dashboard"
CONTRACT_DIR = PUBLIC_BASE / "public_contract"


def read_csv_optional(paths, required=False, label=None):
    if isinstance(paths, (str, Path)):
        paths = [paths]
    tried = []
    for p in paths:
        p = Path(p)
        tried.append(str(p))
        if p.exists():
            return pd.read_csv(p), p
    if required:
        raise FileNotFoundError(f"Required CSV not found for {label or 'file'}; tried: {tried}")
    return pd.DataFrame(), None


metrics, metrics_path = read_csv_optional(
    [DASH_DIR / "annual_balance_dashboard_metrics.csv", PUBLIC_BASE / "annual_balance_dashboard_metrics.csv"],
    required=True,
    label="annual balance dashboard metrics",
)

annual_qa, annual_qa_path = read_csv_optional(
    [DASH_DIR / "annual_balance_dashboard_qa.csv", PUBLIC_BASE / "annual_balance_dashboard_qa.csv"],
    label="annual dashboard QA",
)
metric_contract, metric_contract_path = read_csv_optional(
    [CONTRACT_DIR / "metric_contract_frontier.csv", DASH_DIR / "metric_contract_frontier.csv", PUBLIC_BASE / "metric_contract_frontier.csv"],
    label="metric contract frontier",
)
artifact_contracts, artifact_contracts_path = read_csv_optional(
    [PUBLIC_BASE / "artifact_contracts.csv", CONTRACT_DIR / "artifact_contracts.csv"],
    label="artifact contracts",
)
publish_qa, publish_qa_path = read_csv_optional(
    [PUBLIC_BASE / "publish_contract_qa.csv", CONTRACT_DIR / "publish_contract_qa.csv", DASH_DIR / "publish_contract_qa.csv"],
    label="publish QA",
)
manifest_path = PUBLIC_BASE / "manifest.json"
manifest = json.load(open(manifest_path, "r", encoding="utf-8")) if manifest_path.exists() else {}

loaded = pd.DataFrame([
    {"artifact": "annual_balance_dashboard_metrics", "path": str(metrics_path), "rows": len(metrics)},
    {"artifact": "annual_balance_dashboard_qa", "path": str(annual_qa_path) if annual_qa_path else "missing", "rows": len(annual_qa)},
    {"artifact": "metric_contract_frontier", "path": str(metric_contract_path) if metric_contract_path else "missing", "rows": len(metric_contract)},
    {"artifact": "artifact_contracts", "path": str(artifact_contracts_path) if artifact_contracts_path else "missing", "rows": len(artifact_contracts)},
    {"artifact": "publish_contract_qa", "path": str(publish_qa_path) if publish_qa_path else "missing", "rows": len(publish_qa)},
    {"artifact": "manifest.json", "path": str(manifest_path) if manifest_path.exists() else "missing", "rows": len(manifest) if isinstance(manifest, dict) else None},
])
display(Markdown("### Loaded files"))
display(loaded)

### Loaded files

,artifact,path,rows
0,annual_balance_dashboard_metrics,/home/matias/repos/accounting-backend/public/a...,350
1,annual_balance_dashboard_qa,/home/matias/repos/accounting-backend/public/a...,13
2,metric_contract_frontier,/home/matias/repos/accounting-backend/public/a...,28
3,artifact_contracts,/home/matias/repos/accounting-backend/public/a...,15
4,publish_contract_qa,/home/matias/repos/accounting-backend/public/a...,4
5,manifest.json,/home/matias/repos/accounting-backend/public/a...,21


## 1. Normalización de métricas

El backend puede evolucionar, así que esta notebook normaliza columnas esperadas y tolera columnas ausentes.  
Las columnas más importantes son:

```text
metric_id, Currency, period, value, value_status, dimension_name, dimension_value, caveat
```

In [3]:
def normalize_metrics(df):
    x = df.copy()
    defaults = {
        "metric_id": "", "period_grain": "Y", "period": "", "period_start": "", "period_end": "",
        "Currency": "N/A", "value": np.nan, "value_status": "", "flow_or_stock": "",
        "accounting_section": "", "dashboard_section": "", "dimension_name": "", "dimension_value": "",
        "source_table": "", "source_filter": "", "calculation_rule": "", "frontend_suitability": "",
        "public_flag": "", "internal_flag": "", "legacy_flag": "", "validation_status": "",
        "caveat": "", "run_id": "", "as_of_date": "",
    }
    for col, default in defaults.items():
        if col not in x.columns:
            x[col] = default
    x["metric_id"] = x["metric_id"].fillna("").astype(str)
    x["period"] = x["period"].fillna("").astype(str)
    x["Currency"] = x["Currency"].replace("", np.nan).fillna("N/A").astype(str)
    x["dimension_name"] = x["dimension_name"].replace("", np.nan).fillna("").astype(str)
    x["dimension_value"] = x["dimension_value"].replace("", np.nan).fillna("").astype(str)
    x["value"] = pd.to_numeric(x["value"], errors="coerce")
    for col in ["value_status", "caveat", "validation_status", "frontend_suitability"]:
        x[col] = x[col].replace({np.nan: "", "nan": "", "None": ""}).astype(str)
    return x

inspect_metric_values(
    metrics,
    [
        "IS.REVENUE.OPERATING",
        "IS.RENT.TOTAL",
        "IS.RENT.BY_PROPERTY",
        "IS.OPEX.PROPERTY",
        "IS.OPEX.BY_CATEGORY",
        "IS.NET.OPERATING",
    ],
    top=120,
)

metrics = normalize_dashboard_metrics(metrics)

available_years = sorted(
    metrics.loc[
        metrics["period"].str.match(r"^\d{4}$", na=False),
        "period"
    ].unique().tolist()
)

# metrics = normalize_metrics(metrics)
# available_years = sorted([y for y in metrics["period"].dropna().astype(str).unique() if y])
available_currencies = sorted([c for c in metrics["Currency"].dropna().astype(str).unique() if c])
run_ids = sorted([x for x in metrics["run_id"].dropna().astype(str).unique() if x and x.lower() != "nan"])
as_of_dates = sorted([x for x in metrics["as_of_date"].dropna().astype(str).unique() if x and x.lower() != "nan"])

run_summary = pd.DataFrame([
    {"field": "repo_root", "value": str(REPO_ROOT)},
    {"field": "public_base", "value": str(PUBLIC_BASE)},
    {"field": "metrics_rows", "value": len(metrics)},
    {"field": "years", "value": ", ".join(available_years)},
    {"field": "currencies", "value": ", ".join(available_currencies)},
    {"field": "run_ids", "value": ", ".join(run_ids[:5])},
    {"field": "as_of_dates", "value": ", ".join(as_of_dates[:5])},
])
display(run_summary)

rows: 77
non-null values: 77
null values: 0


,metric_id,Currency,period,dimension_name,dimension_value,value,value_status,source_table,caveat
18,IS.NET.OPERATING,ARS,2022.0,NaN,NaN,4361538.00,available,monthly_operating_statement.csv,NaN
19,IS.NET.OPERATING,ARS,2023.0,NaN,NaN,4802541.00,available,monthly_operating_statement.csv,NaN
21,IS.NET.OPERATING,ARS,2024.0,NaN,NaN,5492034.16,available,monthly_operating_statement.csv,NaN
23,IS.NET.OPERATING,ARS,2025.0,NaN,NaN,27956941.26,available,monthly_operating_statement.csv,NaN
25,IS.NET.OPERATING,ARS,2026.0,NaN,NaN,22166085.31,available,monthly_operating_statement.csv,NaN
20,IS.NET.OPERATING,USD,2023.0,NaN,NaN,0.00,available,monthly_operating_statement.csv,NaN
22,IS.NET.OPERATING,USD,2024.0,NaN,NaN,3610.00,available,monthly_operating_statement.csv,NaN
24,IS.NET.OPERATING,USD,2025.0,NaN,NaN,4560.00,available,monthly_operating_statement.csv,NaN
26,IS.NET.OPERATING,USD,2026.0,NaN,NaN,2280.00,available,monthly_operating_statement.csv,NaN
156,IS.OPEX.BY_CATEGORY,ARS,2022.0,semantic_subbucket,legal,37700.00,available,monthly_flow_semantic_split.csv,NaN


,field,value
0,repo_root,/home/matias/repos/accounting-backend
1,public_base,/home/matias/repos/accounting-backend/public/a...
2,metrics_rows,350
3,years,"2022, 2023, 2024, 2025, 2026"
4,currencies,"ARS, N/A, USD"
5,run_ids,20260710T195604Z
6,as_of_dates,2026-07-10


## 2. Run readiness / estado de publicación

Este bloque resume si el bundle público está listo para mostrar. No corrige nada: solamente muestra los checks disponibles.

Comentarios profesionales:

- Un dashboard patrimonial serio no solo muestra números: también muestra salvedades.
- Si la caja no es `frontend_safe`, se informa como `s/d`, no como cero.
- Si hay fallas de cross-currency, no deben ocultarse: indican que alguna tabla está tratando de sumar ARS y USD.

In [4]:
def qa_status_table(*qa_frames):
    rows = []
    for name, df in qa_frames:
        if df is None or df.empty:
            rows.append({"source": name, "check": "file_available", "status": "missing", "detail": ""})
            continue
        work = df.copy()
        check_col = next((c for c in work.columns if c.lower() in {"check", "check_name", "name", "rule"}), None) or work.columns[0]
        status_col = next((c for c in work.columns if c.lower() in {"status", "result", "level"}), None)
        if status_col is None:
            work["_status"] = "present"
            status_col = "_status"
        detail_col = next((c for c in work.columns if c.lower() in {"detail", "message", "notes", "note"}), None)
        if detail_col is None:
            work["_detail"] = ""
            detail_col = "_detail"
        for _, r in work.iterrows():
            rows.append({"source": name, "check": r.get(check_col, ""), "status": r.get(status_col, ""), "detail": r.get(detail_col, "")})
    return pd.DataFrame(rows)

readiness = qa_status_table(("annual_dashboard_qa", annual_qa), ("publish_contract_qa", publish_qa))
display(readiness)

if not readiness.empty:
    status_s = readiness["status"].astype(str).str.lower()
    attention = readiness[status_s.str.contains("fail|error|warning|warn", na=False)].copy()
    if not attention.empty:
        display(Markdown("### Checks requiring attention"))
        display(attention)

,source,check,status,detail
0,annual_dashboard_qa,annual_metrics_use_only_canonical_sources,pass,"sources=['metric_contract_frontier.csv', 'mont..."
1,annual_dashboard_qa,no_raw_stage_d_sources,pass,"sources=['metric_contract_frontier.csv', 'mont..."
2,annual_dashboard_qa,no_legacy_views_as_canonical,pass,"sources=['metric_contract_frontier.csv', 'mont..."
3,annual_dashboard_qa,annual_flows_sum_monthly_flows,pass,flow rules documented
4,annual_dashboard_qa,annual_stocks_use_last_close,fail,stock rules documented
5,annual_dashboard_qa,ratios_use_annual_aggregates,pass,ratio rules documented
6,annual_dashboard_qa,no_cross_currency_aggregation,pass,money metrics carry Currency
7,annual_dashboard_qa,cash_unavailable_without_frontend_safe_rows,pass,"cash_statuses=['not_applicable', 'unavailable']"
8,annual_dashboard_qa,debt_stock_not_mixed_with_flows,pass,debt OPEN/POSITION are stock
9,annual_dashboard_qa,debt_activity_reconciles_or_residual_visible,pass,adjustments and reconciliation metric present


### Checks requiring attention

,source,check,status,detail
4,annual_dashboard_qa,annual_stocks_use_last_close,fail,stock rules documented
16,publish_contract_qa,debt_stock_activity_separated,fail,debt stock/activity contracts are properly sep...


## 3. Helpers de statements

La notebook usa una lista editable de `line_specs`. Cada línea define qué métrica mostrar, cómo expandir dimensiones y qué comentario profesional incluir.

Convención visual:

```text
Section | Line | Currency | 2022 | 2023 | ... | Professional comment
```

`Currency` queda junto a la línea. ARS y USD no se mezclan.

In [5]:
def is_total_dimension(name, value):
    n = str(name or "").strip().lower()
    v = str(value or "").strip().lower()
    return (not n and not v) or n in {"total", "none", "n/a", "nan"} or v in {"total", "none", "n/a", "nan"}


def select_rows_for_spec(df, spec):
    x = df.copy()
    metric_id = spec.get("metric_id")
    if metric_id:
        x = x[x["metric_id"].eq(metric_id)]
    if x.empty:
        return x
    dim_policy = spec.get("dimension_policy", "total_only")
    if dim_policy == "total_only":
        total_mask = x.apply(lambda r: is_total_dimension(r.get("dimension_name"), r.get("dimension_value")), axis=1)
        return x[total_mask] if total_mask.any() else x
    if dim_policy == "expand":
        return x
    if dim_policy == "filter":
        dn = spec.get("dimension_name")
        dv = spec.get("dimension_value")
        if dn is not None:
            x = x[x["dimension_name"].astype(str).eq(str(dn))]
        if dv is not None:
            x = x[x["dimension_value"].astype(str).eq(str(dv))]
        return x
    return x


def dimension_suffix(row):
    dn = str(row.get("dimension_name", "") or "").strip()
    dv = str(row.get("dimension_value", "") or "").strip()
    if dn and dv and dn.lower() not in {"nan", "none", "n/a"} and dv.lower() not in {"nan", "none", "n/a"}:
        return f"{dn}: {dv}"
    if dv and dv.lower() not in {"nan", "none", "n/a", "total"}:
        return dv
    return ""


def expand_line_label(spec, row):
    base = spec.get("line", spec.get("metric_id", ""))
    if spec.get("dimension_policy") == "expand":
        suffix = dimension_suffix(row)
        if suffix:
            return f"{base} — {suffix}"
    return base

def build_statement_long(df, specs):
    rows = []

    for spec in specs:
        x = df.copy()

        metric_id = spec.get("metric_id")
        if metric_id:
            x = x[x["metric_id"].astype(str).eq(str(metric_id))]

        dim_name = spec.get("dimension_name")
        dim_value = spec.get("dimension_value")

        if dim_name is not None:
            x = x[x["dimension_name"].astype(str).eq(str(dim_name))]

        if dim_value is not None:
            x = x[x["dimension_value"].astype(str).eq(str(dim_value))]

        # Optional value_status filtering
        allowed_statuses = spec.get("allowed_value_statuses")
        if allowed_statuses is not None and "value_status" in x.columns:
            x = x[x["value_status"].astype(str).isin(allowed_statuses)]

        # If no rows matched, emit one visible placeholder row.
        if x.empty:
            rows.append({
                "statement": spec.get("statement", ""),
                "section": spec.get("section", ""),
                "line_order": spec.get("order", 9999),
                "line": spec.get("line", metric_id or "missing metric"),
                "metric_id": metric_id or "",
                "Currency": spec.get("Currency", "s/d"),
                "period": "",
                "value": np.nan,
                "value_status": "missing_metric",
                "format_hint": spec.get("format_hint", "number"),
                "professional_comment": spec.get("professional_comment", ""),
                "caveat": spec.get("caveat", ""),
                "source_table": "",
            })
            continue

        for _, r in x.iterrows():
            dimension_name = str(r.get("dimension_name", "") or "")
            dimension_value = str(r.get("dimension_value", "") or "")

            line = spec.get("line", metric_id)

            # For dimensioned metrics, append useful dimension label unless spec disables it.
            if spec.get("append_dimension", True):
                if dimension_name and dimension_name.lower() not in ["nan", "none", ""]:
                    if dimension_value and dimension_value.lower() not in ["nan", "none", ""]:
                        if dimension_value not in line:
                            line = f"{line} — {dimension_name}: {dimension_value}"

            rows.append({
                "statement": spec.get("statement", ""),
                "section": spec.get("section", ""),
                "line_order": spec.get("order", 9999),
                "line": line,
                "metric_id": metric_id or r.get("metric_id", ""),
                "Currency": r.get("Currency", "N/A"),
                "period": str(r.get("period", "")).replace(".0", ""),
                "value": pd.to_numeric(r.get("value", np.nan), errors="coerce"),
                "value_status": r.get("value_status", ""),
                "format_hint": spec.get("format_hint", "number"),
                "professional_comment": spec.get("professional_comment", ""),
                "caveat": r.get("caveat", spec.get("caveat", "")),
                "source_table": r.get("source_table", ""),
            })

    return pd.DataFrame(rows)

def agg_value(series):
    vals = pd.to_numeric(series, errors="coerce").dropna()
    return np.nan if len(vals) == 0 else vals.sum()



def build_statement_table(df, specs, years=None, diagnose=False, top=40):
    """
    Drop-in replacement for build_statement_table with diagnostics.

    Main fixes:
    - Avoids pivot_table(..., dropna=False), which can create unobserved cartesian combinations.
    - Uses groupby + unstack to preserve only observed keys.
    - Shows diagnostics for duplicated keys, display collisions, all-NaN rows, and merge cardinality.
    """
    years = years or available_years
    years = [str(y).replace(".0", "") for y in years]

    long = build_statement_long(df, specs)#.copy()

    # Normalize period labels, because columns may be 2022.0 / 2022 / "2022"
    if "period" in long.columns:
        long["period"] = long["period"].astype(str).str.strip().str.replace(r"\.0$", "", regex=True)

    id_cols = [
        "statement",
        "section",
        "line_order",
        "line",
        "metric_id",
        "Currency",
        "format_hint",
        "professional_comment",
    ]
    id_cols = [c for c in id_cols if c in long.columns]

    # Keep period rows only
    value_long = long[long["period"].astype(str).ne("")].copy()

    # Coerce numeric value
    if "value" in value_long.columns:
        value_long["value"] = pd.to_numeric(value_long["value"], errors="coerce")

    # ---------------------------------------------------------------------
    # Diagnostics: raw long table
    # ---------------------------------------------------------------------
    if diagnose:
        display(Markdown("## build_statement_table diagnostics"))

        display(Markdown("### 1. Long table shape"))
        display(pd.DataFrame([{
            "long_rows": len(long),
            "value_long_rows": len(value_long),
            "n_specs": len(specs),
            "n_id_cols": len(id_cols),
            "years": ", ".join(years),
        }]))

        display(Markdown("### 2. Long columns"))
        display(pd.DataFrame({"column": list(long.columns)}))

        # Rows where line label collides visually
        display_keys = [c for c in ["section", "line", "Currency"] if c in long.columns]
        if display_keys:
            collisions = (
                value_long
                .groupby(display_keys, dropna=False)
                .agg(
                    n_rows=("metric_id", "size"),
                    n_metric_ids=("metric_id", lambda s: s.dropna().astype(str).nunique()),
                    metric_ids=("metric_id", lambda s: ", ".join(sorted(s.dropna().astype(str).unique())[:12])),
                    periods=("period", lambda s: ", ".join(sorted(s.dropna().astype(str).unique())[:12])),
                    comments=("professional_comment", lambda s: " | ".join(pd.Series(s).dropna().astype(str).unique()[:4])),
                )
                .reset_index()
                .sort_values(["n_rows", "n_metric_ids"], ascending=False)
            )
            collisions = collisions[(collisions["n_rows"] > 1) | (collisions["n_metric_ids"] > 1)]

            display(Markdown("### 3. Display collisions: rows that look duplicated once metric_id is hidden"))
            display(collisions.head(top))

        # Key-period multiplicity
        key_period_cols = id_cols + ["period"]
        key_counts = (
            value_long
            .groupby(key_period_cols, dropna=False)
            .agg(
                n_rows=("value", "size"),
                n_non_null_values=("value", lambda s: s.notna().sum()),
                n_distinct_values=("value", lambda s: s.dropna().nunique()),
                values_sample=("value", lambda s: ", ".join(map(str, s.dropna().head(8).tolist()))),
            )
            .reset_index()
            .sort_values(["n_rows", "n_distinct_values"], ascending=False)
        )

        suspicious_keys = key_counts[
            (key_counts["n_rows"] > 1) | (key_counts["n_distinct_values"] > 1)
        ]

        display(Markdown("### 4. Multiple rows for same id_cols + period"))
        display(suspicious_keys.head(top))

        # Missing metric rows
        missing = value_long[value_long["value"].isna()].copy()
        display(Markdown("### 5. Rows with NaN value before pivot"))
        display(
            missing[
                [c for c in ["statement", "section", "line", "metric_id", "Currency", "period", "value_status", "caveat", "source_table"] if c in missing.columns]
            ].head(top)
        )

    # ---------------------------------------------------------------------
    # Build wide safely
    # ---------------------------------------------------------------------
    if value_long.empty:
        wide = long[id_cols + [c for c in ["value_status", "caveat", "source_table"] if c in long.columns]].drop_duplicates()
        for y in years:
            wide[y] = np.nan
    else:
        grouped = (
            value_long
            .groupby(id_cols + ["period"], dropna=False, as_index=False)
            .agg(value=("value", agg_value))
        )

        # Safer than pivot_table(dropna=False): only observed combinations survive.
        wide = (
            grouped
            .set_index(id_cols + ["period"])["value"]
            .unstack("period")
            .reset_index()
        )

        for y in years:
            if y not in wide.columns:
                wide[y] = np.nan

    # ---------------------------------------------------------------------
    # Build meta
    # ---------------------------------------------------------------------
    meta = (
        long.groupby(id_cols, dropna=False)
        .agg(
            value_status=("value_status", lambda s: ", ".join(sorted({str(x) for x in s if str(x) and str(x) != "nan"}))[:240] if "value_status" in long.columns else ""),
            caveat=("caveat", lambda s: " | ".join([str(x) for x in s if str(x) and str(x) != "nan"][:3]) if "caveat" in long.columns else ""),
            source_table=("source_table", lambda s: ", ".join(sorted({str(x) for x in s if str(x) and str(x) != "nan"}))[:240] if "source_table" in long.columns else ""),
        )
        .reset_index()
    )

    if diagnose:
        display(Markdown("### 6. Wide shape before meta merge"))
        display(pd.DataFrame([{
            "wide_rows": len(wide),
            "meta_rows": len(meta),
            "unique_wide_keys": len(wide[id_cols].drop_duplicates()) if id_cols else len(wide),
            "unique_meta_keys": len(meta[id_cols].drop_duplicates()) if id_cols else len(meta),
        }]))

        meta_dups = meta[meta.duplicated(id_cols, keep=False)].sort_values(id_cols)
        display(Markdown("### 7. Meta duplicate keys"))
        display(meta_dups.head(top))

    # Validate merge cardinality
    try:
        wide = wide.merge(meta, on=id_cols, how="left", validate="one_to_one")
        merge_status = "one_to_one"
    except Exception as e:
        merge_status = f"merge validation failed: {e}"
        wide = wide.merge(meta, on=id_cols, how="left")

    if diagnose:
        display(Markdown("### 8. Merge status"))
        display(pd.DataFrame([{"merge_status": merge_status}]))

    # Sort
    sort_cols = [c for c in ["statement", "section", "line_order", "line", "Currency", "metric_id"] if c in wide.columns]
    if sort_cols:
        wide = wide.sort_values(sort_cols)

    # ---------------------------------------------------------------------
    # Diagnostics after final wide
    # ---------------------------------------------------------------------
    if diagnose:
        year_cols = [y for y in years if y in wide.columns]

        all_nan_rows = wide[wide[year_cols].isna().all(axis=1)].copy() if year_cols else pd.DataFrame()
        display(Markdown("### 9. Final rows where all year values are NaN"))
        display(
            all_nan_rows[
                [c for c in ["statement", "section", "line", "metric_id", "Currency", "value_status", "caveat", "source_table"] if c in all_nan_rows.columns]
            ].head(top)
        )

        final_display_keys = [c for c in ["section", "line", "Currency"] if c in wide.columns]
        if final_display_keys:
            final_collisions = (
                wide
                .groupby(final_display_keys, dropna=False)
                .agg(
                    n_rows=("metric_id", "size"),
                    metric_ids=("metric_id", lambda s: ", ".join(sorted(pd.Series(s).dropna().astype(str).unique())[:20])),
                    comments=("professional_comment", lambda s: " | ".join(pd.Series(s).dropna().astype(str).unique()[:6])),
                    all_years_nan=("metric_id", lambda s: False),
                )
                .reset_index()
                .sort_values("n_rows", ascending=False)
            )
            final_collisions = final_collisions[final_collisions["n_rows"] > 1]

            display(Markdown("### 10. Final display collisions"))
            display(final_collisions.head(top))

        display(Markdown("### 11. Final wide preview with metric_id visible"))
        preview_cols = [c for c in ["section", "line", "metric_id", "Currency", *years, "value_status", "professional_comment", "caveat", "source_table"] if c in wide.columns]
        display(wide[preview_cols].head(top))

    # ---------------------------------------------------------------------
    # Final display columns
    # ---------------------------------------------------------------------
    display_cols = [
        "section",
        "line",
        "Currency",
        *years,
        "value_status",
        "professional_comment",
        "caveat",
        "format_hint",
    ]

    # During diagnosis, include metric_id to reveal apparent duplicates.
    if diagnose and "metric_id" in wide.columns:
        display_cols = [
            "section",
            "line",
            "metric_id",
            "Currency",
            *years,
            "value_status",
            "professional_comment",
            "caveat",
            "format_hint",
            "source_table",
        ]

    return wide[[c for c in display_cols if c in wide.columns]].copy()


def format_int_ar(value):
    """
    1234567.89 -> '1.234.568'
    No cents. Argentine thousands separator.
    """
    if pd.isna(value):
        return "s/d"

    try:
        v = round(float(value))
    except Exception:
        return str(value)

    return f"{v:,.0f}".replace(",", ".")


def format_percent_ar(value):
    """
    0.273 -> '27,3%'
    27.3  -> '27,3%'
    """
    if pd.isna(value):
        return "s/d"

    try:
        v = float(value)
    except Exception:
        return str(value)

    shown = v * 100 if abs(v) <= 2 else v
    return f"{shown:.1f}".replace(".", ",") + "%"


def format_value(value, currency=None, format_hint=None):
    if pd.isna(value):
        return "s/d"

    fmt = str(format_hint or "").lower()

    if fmt in {"percent", "ratio", "rate"}:
        return format_percent_ar(value)

    return format_int_ar(value)
    
def display_statement(df, title, subtitle=None):
    display(Markdown(f"## {title}"))
    if subtitle:
        display(Markdown(subtitle))

    out = df.copy()

    year_cols = [
        c for c in out.columns
        if str(c).replace(".0", "").isdigit()
    ]

    # Normalize year column labels: 2022.0 -> 2022
    rename_years = {
        c: str(c).replace(".0", "")
        for c in year_cols
    }
    out = out.rename(columns=rename_years)
    year_cols = [rename_years[c] for c in year_cols]

    # Important: make display columns object before assigning strings.
    for y in year_cols:
        out[y] = out[y].astype("object")

    for idx, row in out.iterrows():
        for y in year_cols:
            out.at[idx, y] = format_value(
                row[y],
                row.get("Currency"),
                row.get("format_hint", "")
            )

    display_cols = [c for c in out.columns if c not in {"format_hint"}]
    out = out[display_cols]

    display(out.style.hide(axis="index"))

def metric_inventory(metric_prefix=None):
    x = metrics.copy()
    if metric_prefix:
        x = x[x["metric_id"].astype(str).str.startswith(metric_prefix)]
    cols = ["metric_id", "dashboard_section", "dimension_name", "dimension_value", "Currency", "period", "value_status", "source_table"]
    cols = [c for c in cols if c in x.columns]
    return x[cols].drop_duplicates().sort_values([c for c in ["metric_id", "dimension_name", "dimension_value", "Currency", "period"] if c in cols])

## Estado de resultado operativo


Comentario profesional:

- Este cuadro responde si la operación de propiedades genera resultado por sí misma.
- No incluye aportes, préstamos, repagos, dividendos ni gasto personal.
- `IS.NET.OPERATING` es la métrica limpia para resultado operativo.
- Si aparecen líneas USD acá, se muestran integradas, pero no se suman con ARS.

In [6]:
operating_specs = [
    {
        "statement": "Estado de resultado operativo",
        "section": "Ingresos operativos",
        "line": "Ingresos operativos",
        "metric_id": "IS.REVENUE.OPERATING",
        "dimension_policy": "total_only",
        "order": 100,
        "comment": "Ingreso operativo reconocido por la actividad patrimonial. Debe mantenerse separado de aportes, deuda y funding."
    },
    {
        "statement": "Estado de resultado operativo",
        "section": "Ingresos operativos",
        "line": "Renta total",
        "metric_id": "IS.RENT.TOTAL",
        "dimension_policy": "total_only",
        "order": 110,
        "comment": "Renta/alquiler total del año. Es revenue operativo, no incluye contribuciones familiares."
    },
    {
        "statement": "Estado de resultado operativo",
        "section": "Ingresos operativos",
        "line": "Renta por propiedad",
        "metric_id": "IS.RENT.BY_PROPERTY",
        "dimension_policy": "expand",
        "order": 120,
        "comment": "Apertura por propiedad/lugar. Sirve para detectar qué activo genera la renta."
    },
    {
        "statement": "Estado de resultado operativo",
        "section": "Costos operativos de propiedad",
        "line": "OPEX propiedad total",
        "metric_id": "IS.OPEX.PROPERTY",
        "dimension_policy": "total_only",
        "order": 200,
        "comment": "Costos reales de operar/mantener propiedades: impuestos, servicios, mantenimiento, legal y otros OPEX patrimoniales."
    },
    {
        "statement": "Estado de resultado operativo",
        "section": "Costos operativos de propiedad",
        "line": "OPEX por categoría",
        "metric_id": "IS.OPEX.BY_CATEGORY",
        "dimension_policy": "expand",
        "order": 210,
        "comment": "Apertura profesional de OPEX. No debe incluir deuda, repagos, retiros o gasto personal."
    },
    {
        "statement": "Estado de resultado operativo",
        "section": "Resultado",
        "line": "Resultado operativo neto",
        "metric_id": "IS.NET.OPERATING",
        "dimension_policy": "total_only",
        "order": 300,
        "comment": "Resultado operativo limpio: ingresos operativos menos OPEX propiedad. Es la métrica más cercana al resultado económico de la operación."
    }
]

operating_table = build_statement_table(metrics, operating_specs).drop_duplicates()
display_statement(operating_table, "Estado de resultado operativo", "Ingresos y costos propios de la operación patrimonial. No incluye funding, retiros ni deuda.")

## Estado de resultado operativo

Ingresos y costos propios de la operación patrimonial. No incluye funding, retiros ni deuda.

section,line,Currency,2022,2023,2024,2025,2026,value_status,professional_comment,caveat
Costos operativos de propiedad,OPEX propiedad total,ARS,587.266,4.158.761,8.066.302,12.168.481,7.422.915,available,,
Costos operativos de propiedad,OPEX propiedad total,USD,s/d,0,0,0,0,available,,
Costos operativos de propiedad,OPEX por categoría — semantic_subbucket: legal,ARS,37.700,2.043.313,1.044.210,147.000,100.000,available,,
Costos operativos de propiedad,OPEX por categoría — semantic_subbucket: maintenance,ARS,215.900,57.500,592.174,1.131.802,134.831,available,,
Costos operativos de propiedad,OPEX por categoría — semantic_subbucket: services,ARS,105.726,516.242,2.246.508,4.012.698,1.773.488,available,,
Costos operativos de propiedad,OPEX por categoría — semantic_subbucket: taxes,ARS,227.940,1.541.706,4.183.410,6.876.980,5.414.596,available,,
Ingresos operativos,Ingresos operativos,ARS,4.948.804,8.961.302,13.558.336,40.125.422,29.589.000,available,,
Ingresos operativos,Ingresos operativos,USD,s/d,0,3.610,4.560,2.280,available,,
Ingresos operativos,Renta total,ARS,4.948.804,8.961.302,13.558.336,40.125.422,29.589.000,available,,
Ingresos operativos,Renta total,USD,s/d,s/d,3.610,4.560,2.280,available,,


## Puente de fondos y distribuciones


Comentario profesional:

- Este cuadro no es un income statement puro; es un **funds bridge**.
- Muestra cómo el resultado operativo se complementa con aportes/funding y cómo se erosiona con retiros/distribuciones.
- No conviene llamar “profit” a `COV.NET.AFTER_DRAWS` sin salvedad, porque mezcla operación, funding y distribuciones.

In [7]:
funds_bridge_specs = [
    {
        "statement": "Puente de fondos y distribuciones",
        "section": "Base operativa",
        "line": "Resultado operativo neto",
        "metric_id": "IS.NET.OPERATING",
        "dimension_policy": "total_only",
        "order": 100,
        "comment": "Punto de partida del puente. Mide la operación antes de aportes y retiros."
    },
    {
        "statement": "Puente de fondos y distribuciones",
        "section": "Funding / aportes",
        "line": "Contribuciones / funding total",
        "metric_id": "FUND.CONTRIB.TOTAL",
        "dimension_policy": "total_only",
        "order": 200,
        "comment": "Aportes o funding que cubren necesidades de la órbita. No son revenue operativo."
    },
    {
        "statement": "Puente de fondos y distribuciones",
        "section": "Funding / aportes",
        "line": "Funding por actor",
        "metric_id": "FUND.CONTRIB.BY_ACTOR",
        "dimension_policy": "expand",
        "order": 210,
        "comment": "Apertura por actor que aportó fondos. Útil para conversaciones familiares y gobernanza."
    },
    {
        "statement": "Puente de fondos y distribuciones",
        "section": "Distribuciones / retiros",
        "line": "Retiros / gasto personal",
        "metric_id": "DIST.DRAWS.PERSONAL",
        "dimension_policy": "total_only",
        "order": 300,
        "comment": "Uso personal/familiar de fondos. No debe mezclarse con OPEX de propiedad."
    },
    {
        "statement": "Puente de fondos y distribuciones",
        "section": "Distribuciones / retiros",
        "line": "Dividendos",
        "metric_id": "DIST.DIVIDENDS",
        "dimension_policy": "total_only",
        "order": 310,
        "comment": "Distribuciones explícitas. Se muestran separadas de gasto operativo."
    },
    {
        "statement": "Puente de fondos y distribuciones",
        "section": "Distribuciones / retiros",
        "line": "Retiros por tipo",
        "metric_id": "DIST.DRAWS.BY_TYPE",
        "dimension_policy": "expand",
        "order": 320,
        "comment": "Apertura de retiros/distribuciones por tipo, si el backend la produce."
    },
    {
        "statement": "Puente de fondos y distribuciones",
        "section": "Cobertura",
        "line": "Cobertura después de funding y retiros",
        "metric_id": "COV.NET.AFTER_DRAWS",
        "dimension_policy": "total_only",
        "order": 400,
        "comment": "Resultado de cobertura gerencial: resultado operativo + funding - retiros/distribuciones."
    },
    {
        "statement": "Puente de fondos y distribuciones",
        "section": "Indicadores",
        "line": "Savings / coverage rate",
        "metric_id": "COV.SAVINGS_RATE",
        "dimension_policy": "total_only",
        "order": 500,
        "comment": "Indicador de cobertura. Debe calcularse desde agregados anuales, no promediando ratios mensuales.",
        "format_hint": "percent"
    }
]

funds_bridge_table = build_statement_table(metrics, funds_bridge_specs)
display_statement(funds_bridge_table, "Puente de fondos y distribuciones", "Muestra aportes, retiros y cobertura después del resultado operativo.")

## Puente de fondos y distribuciones

Muestra aportes, retiros y cobertura después del resultado operativo.

section,line,Currency,2022,2023,2024,2025,2026,value_status,professional_comment,caveat
Base operativa,Resultado operativo neto,ARS,4.361.538,4.802.541,5.492.034,27.956.941,22.166.085,available,,
Base operativa,Resultado operativo neto,USD,s/d,0,3.610,4.560,2.280,available,,
Cobertura,Cobertura después de funding y retiros,ARS,0,-1.779.785,-4.754.302,-2.836.747,-4.239.925,available,,
Cobertura,Cobertura después de funding y retiros,USD,s/d,0,3.510,4.560,1.900,available,,
Distribuciones / retiros,Retiros / gasto personal,ARS,4.361.538,8.588.546,13.558.336,34.485.689,26.516.010,available,,
Distribuciones / retiros,Retiros / gasto personal,USD,s/d,0,100,0,380,available,,
Distribuciones / retiros,Dividendos,ARS,0,0,0,0,1.367.384,available,,
Distribuciones / retiros,Dividendos,USD,s/d,0,100,0,380,available,,
Distribuciones / retiros,Retiros por tipo — semantic_subbucket: dividend,ARS,s/d,s/d,s/d,s/d,1.367.384,available,,
Distribuciones / retiros,Retiros por tipo — semantic_subbucket: dividend,USD,s/d,s/d,100,s/d,380,available,,


## Canales de pago / settlement


Comentario profesional:

- Este cuadro busca separar **obligación económica** de **canal de liquidación**.
- En tu caso, `Box` es una órbita de gobernanza, no siempre una caja física.
- Inquilinos o actores pueden pagar directo a impuestos/servicios sin pasar por PM/FB.
- Si estas métricas todavía no existen en el annual layer, la tabla mostrará `missing_metric`; eso señala una mejora de backend, no necesariamente un problema de datos.

In [8]:
settlement_specs = [
    {
        "statement": "Canales de pago / settlement",
        "section": "Pagos por caja gobernada",
        "line": "Pagos vía caja PM/FB",
        "metric_id": "SETTLEMENT.BOX_CASH.OUTFLOWS",
        "dimension_policy": "expand",
        "order": 100,
        "comment": "Obligaciones pagadas desde caja real/gobernada. Solo debería usarse si cash_path confirma caja."
    },
    {
        "statement": "Canales de pago / settlement",
        "section": "Pagos directos de terceros",
        "line": "Pagos directos de inquilinos",
        "metric_id": "SETTLEMENT.TENANT_DIRECT_PAYMENTS",
        "dimension_policy": "expand",
        "order": 200,
        "comment": "Pagos de inquilinos directamente a impuestos/servicios/proveedores. Son settlement directo, no caja PM."
    },
    {
        "statement": "Canales de pago / settlement",
        "section": "Pagos directos de terceros",
        "line": "Pagos directos de actores",
        "metric_id": "SETTLEMENT.ACTOR_DIRECT_PAYMENTS",
        "dimension_policy": "expand",
        "order": 210,
        "comment": "Pagos de Matías/Alejandro/Héctor/Primos en nombre de una órbita. Pueden crear claims internos."
    },
    {
        "statement": "Canales de pago / settlement",
        "section": "Reembolsos y actor-to-actor",
        "line": "Settlements entre actores",
        "metric_id": "SETTLEMENT.ACTOR_TO_ACTOR",
        "dimension_policy": "expand",
        "order": 300,
        "comment": "Transferencias entre actores para regularizar obligaciones subyacentes. No deben duplicar OPEX ya reconocido."
    },
    {
        "statement": "Canales de pago / settlement",
        "section": "No-cash",
        "line": "Devengamientos no-cash",
        "metric_id": "SETTLEMENT.NON_CASH_ACCRUALS",
        "dimension_policy": "expand",
        "order": 400,
        "comment": "Intereses, refinanciaciones o reconocimientos que crean deuda sin movimiento de caja."
    }
]

settlement_table = build_statement_table(metrics, settlement_specs)
display_statement(settlement_table, "Canales de pago / settlement", "Mapa gerencial de cómo se liquidan obligaciones: caja, pago directo, reembolso, actor-to-actor o no-cash.")

## Canales de pago / settlement

Mapa gerencial de cómo se liquidan obligaciones: caja, pago directo, reembolso, actor-to-actor o no-cash.

section,line,Currency,2022,2023,2024,2025,2026,professional_comment
No-cash,Devengamientos no-cash,s/d,s/d,s/d,s/d,s/d,s/d,
Pagos directos de terceros,Pagos directos de inquilinos,s/d,s/d,s/d,s/d,s/d,s/d,
Pagos directos de terceros,Pagos directos de actores,s/d,s/d,s/d,s/d,s/d,s/d,
Pagos por caja gobernada,Pagos vía caja PM/FB,s/d,s/d,s/d,s/d,s/d,s/d,
Reembolsos y actor-to-actor,Settlements entre actores,s/d,s/d,s/d,s/d,s/d,s/d,


## Estado de posición operativa


Comentario profesional:

- Este cuadro se parece al balance, pero no es un balance legal completo.
- No incluye valuación de propiedades.
- Caja solo se muestra si es `frontend_safe`.
- Deuda y claims se muestran por moneda, sin convertir ARS/USD.
- La posición neta interna solo debe interpretarse con salvedades si caja validada no existe.

In [9]:
position_specs = [
    {
        "statement": "Estado de posición operativa",
        "section": "Caja / activos líquidos",
        "line": "Caja total validada",
        "metric_id": "BS.CASH.TOTAL",
        "dimension_policy": "total_only",
        "order": 100,
        "comment": "Caja mostrable solo si proviene de fuente frontend-safe. Si no, debe ser s/d, no cero."
    },
    {
        "statement": "Estado de posición operativa",
        "section": "Caja / activos líquidos",
        "line": "Caja por box/cuenta",
        "metric_id": "BS.CASH.CLOSE.BOX",
        "dimension_policy": "expand",
        "order": 110,
        "comment": "Apertura de caja por box/cuenta. Box de gobernanza no equivale automáticamente a caja física."
    },
    {
        "statement": "Estado de posición operativa",
        "section": "Caja / activos líquidos",
        "line": "Depósitos de garantía retenidos",
        "metric_id": "BS.SECURITY_DEPOSITS.HELD",
        "dimension_policy": "total_only",
        "order": 120,
        "comment": "Depósitos o garantías si el backend los modela. Si falta, queda como s/d."
    },
    {
        "statement": "Estado de posición operativa",
        "section": "Claims / cuentas por cobrar",
        "line": "Claims abiertos por contraparte",
        "metric_id": "ID.DEBT.OPEN.BY_COUNTERPARTY",
        "dimension_policy": "expand",
        "order": 200,
        "comment": "Stock de deuda/claim abierto por deudor-acreedor. Leer siempre como debtor debe a creditor."
    },
    {
        "statement": "Estado de posición operativa",
        "section": "Deuda interna",
        "line": "Deuda total abierta",
        "metric_id": "ID.DEBT.TOTAL.OPEN",
        "dimension_policy": "total_only",
        "order": 300,
        "comment": "Stock final anual de deuda interna abierta, por moneda. No es flow."
    },
    {
        "statement": "Estado de posición operativa",
        "section": "Deuda interna",
        "line": "Principal abierto",
        "metric_id": "ID.DEBT.PRINCIPAL.OPEN",
        "dimension_policy": "total_only",
        "order": 310,
        "comment": "Componente principal de deuda abierta."
    },
    {
        "statement": "Estado de posición operativa",
        "section": "Deuda interna",
        "line": "Interés abierto",
        "metric_id": "ID.DEBT.INTEREST.OPEN",
        "dimension_policy": "total_only",
        "order": 320,
        "comment": "Interés abierto/devengado aún no cancelado."
    },
    {
        "statement": "Estado de posición operativa",
        "section": "Neto interno",
        "line": "Posición neta PM",
        "metric_id": "ID.DEBT.NET_PM_POSITION",
        "dimension_policy": "total_only",
        "order": 400,
        "comment": "Vista neta interna de PM si el backend la produce. Interpretar por moneda y con criterio de deudor/acreedor."
    }
]

position_table = build_statement_table(metrics, position_specs)
display_statement(position_table, "Estado de posición operativa", "Stocks anuales: caja validada, claims y deuda interna. No mezcla stocks con flows.")

## Estado de posición operativa

Stocks anuales: caja validada, claims y deuda interna. No mezcla stocks con flows.

section,line,Currency,2022,2023,2024,2025,2026,value_status,professional_comment,caveat
Caja / activos líquidos,Caja total validada,N/A,s/d,s/d,s/d,s/d,s/d,unavailable,,No frontend-safe cash rows exist; no fallback used.
Caja / activos líquidos,Caja por box/cuenta,N/A,s/d,s/d,s/d,s/d,s/d,unavailable,,No frontend-safe cash rows exist; no fallback used.
Claims / cuentas por cobrar,Claims abiertos por contraparte — debtor_creditor: Alejandro -> MI,USD,s/d,s/d,253,334,418,available,,
Claims / cuentas por cobrar,Claims abiertos por contraparte — debtor_creditor: Alejandro -> PM,USD,s/d,6.703,6.703,6.703,6.331,available,,
Claims / cuentas por cobrar,Claims abiertos por contraparte — debtor_creditor: Hector -> MI,USD,s/d,s/d,s/d,490,505,available,,
Claims / cuentas por cobrar,Claims abiertos por contraparte — debtor_creditor: PM -> MI,USD,s/d,5.792,8.536,6.810,2.316,available,,
Claims / cuentas por cobrar,Claims abiertos por contraparte — debtor_creditor: PM -> Primos,USD,s/d,897,897,897,897,available,,
Deuda interna,Deuda total abierta,USD,s/d,13.392,16.389,15.235,10.467,available,,"Debt is stock, not flow; not mixed into operating result. | Debt is stock, not flow; not mixed into operating result. | Debt is stock, not flow; not mixed into operating result."
Deuda interna,Principal abierto,USD,s/d,13.392,16.316,15.081,10.214,available,,"Debt is stock, not flow; not mixed into operating result. | Debt is stock, not flow; not mixed into operating result. | Debt is stock, not flow; not mixed into operating result."
Deuda interna,Interés abierto,USD,s/d,0,73,154,253,available,,"Debt is stock, not flow; not mixed into operating result. | Debt is stock, not flow; not mixed into operating result. | Debt is stock, not flow; not mixed into operating result."


## Movimiento anual de deuda interna


Comentario profesional:

- Este cuadro es un roll-forward de deuda.
- Separa deuda inicial/final, nuevos claims, intereses, repagos y ajustes.
- Repagos de deuda no son OPEX.
- Ajustes residuales deben mostrarse; no se esconden para que cierre artificialmente.

In [10]:
debt_rollforward_specs = [
    {
        "statement": "Movimiento anual de deuda interna",
        "section": "Actividad",
        "line": "Nuevos claims / nuevos préstamos",
        "metric_id": "ID.DEBT.ACTIVITY.NEW_CLAIMS",
        "dimension_policy": "expand",
        "order": 100,
        "comment": "Nuevos derechos/obligaciones creadas durante el año. Flow de deuda, no revenue ni OPEX."
    },
    {
        "statement": "Movimiento anual de deuda interna",
        "section": "Actividad",
        "line": "Intereses devengados",
        "metric_id": "ID.DEBT.ACTIVITY.INTEREST_ACCRUED",
        "dimension_policy": "expand",
        "order": 110,
        "comment": "Costo de oportunidad/interés reconocido como actividad de deuda. No es OPEX de propiedad."
    },
    {
        "statement": "Movimiento anual de deuda interna",
        "section": "Actividad",
        "line": "Repagos",
        "metric_id": "ID.DEBT.ACTIVITY.REPAYMENTS",
        "dimension_policy": "expand",
        "order": 120,
        "comment": "Cancelaciones de deuda. No deben aparecer como costo operativo."
    },
    {
        "statement": "Movimiento anual de deuda interna",
        "section": "Actividad",
        "line": "Ajustes residuales",
        "metric_id": "ID.DEBT.ACTIVITY.ADJUSTMENTS",
        "dimension_policy": "expand",
        "order": 130,
        "comment": "Diferencias o residuos visibles. Sirven para auditar el resolver y no deben ocultarse."
    },
    {
        "statement": "Movimiento anual de deuda interna",
        "section": "Actividad",
        "line": "Cambio neto de deuda",
        "metric_id": "ID.DEBT.ACTIVITY.NET_CHANGE",
        "dimension_policy": "expand",
        "order": 140,
        "comment": "Cambio neto anual explicado por claims, intereses, repagos y ajustes."
    },
    {
        "statement": "Movimiento anual de deuda interna",
        "section": "Stock final",
        "line": "Deuda final abierta",
        "metric_id": "ID.DEBT.TOTAL.OPEN",
        "dimension_policy": "total_only",
        "order": 200,
        "comment": "Stock final anual de deuda. Debe reconciliar contra la actividad anual."
    }
]

debt_rollforward_table = build_statement_table(metrics, debt_rollforward_specs)
display_statement(debt_rollforward_table, "Movimiento anual de deuda interna", "Roll-forward de deuda: actividad del año y stock final.")

## Movimiento anual de deuda interna

Roll-forward de deuda: actividad del año y stock final.

section,line,Currency,2022,2023,2024,2025,2026,value_status,professional_comment,caveat
Actividad,Nuevos claims / nuevos préstamos — debtor_creditor: Alejandro -> MI,USD,s/d,s/d,180,0,0,available,,Debt movement; not OPEX or funding unless classified elsewhere. | Debt movement; not OPEX or funding unless classified elsewhere. | Debt movement; not OPEX or funding unless classified elsewhere.
Actividad,Nuevos claims / nuevos préstamos — debtor_creditor: Alejandro -> PM,USD,s/d,6.703,0,0,0,available,,Debt movement; not OPEX or funding unless classified elsewhere. | Debt movement; not OPEX or funding unless classified elsewhere. | Debt movement; not OPEX or funding unless classified elsewhere.
Actividad,Nuevos claims / nuevos préstamos — debtor_creditor: Hector -> MI,USD,s/d,s/d,s/d,490,0,available,,Debt movement; not OPEX or funding unless classified elsewhere. | Debt movement; not OPEX or funding unless classified elsewhere.
Actividad,Nuevos claims / nuevos préstamos — debtor_creditor: PM -> MI,USD,s/d,5.806,2.750,268,0,available,,Debt movement; not OPEX or funding unless classified elsewhere. | Debt movement; not OPEX or funding unless classified elsewhere. | Debt movement; not OPEX or funding unless classified elsewhere.
Actividad,Nuevos claims / nuevos préstamos — debtor_creditor: PM -> Primos,USD,s/d,897,0,0,0,available,,Debt movement; not OPEX or funding unless classified elsewhere. | Debt movement; not OPEX or funding unless classified elsewhere. | Debt movement; not OPEX or funding unless classified elsewhere.
Actividad,Intereses devengados — debtor_creditor: Alejandro -> MI,USD,s/d,s/d,73,81,84,available,,Debt movement; not OPEX or funding unless classified elsewhere. | Debt movement; not OPEX or funding unless classified elsewhere. | Debt movement; not OPEX or funding unless classified elsewhere.
Actividad,Intereses devengados — debtor_creditor: Alejandro -> PM,USD,s/d,0,0,0,0,available,,Debt movement; not OPEX or funding unless classified elsewhere. | Debt movement; not OPEX or funding unless classified elsewhere. | Debt movement; not OPEX or funding unless classified elsewhere.
Actividad,Intereses devengados — debtor_creditor: Hector -> MI,USD,s/d,s/d,s/d,0,15,available,,Debt movement; not OPEX or funding unless classified elsewhere. | Debt movement; not OPEX or funding unless classified elsewhere.
Actividad,Intereses devengados — debtor_creditor: PM -> MI,USD,s/d,0,101,187,200,available,,Debt movement; not OPEX or funding unless classified elsewhere. | Debt movement; not OPEX or funding unless classified elsewhere. | Debt movement; not OPEX or funding unless classified elsewhere.
Actividad,Intereses devengados — debtor_creditor: PM -> Primos,USD,s/d,0,0,0,0,available,,Debt movement; not OPEX or funding unless classified elsewhere. | Debt movement; not OPEX or funding unless classified elsewhere. | Debt movement; not OPEX or funding unless classified elsewhere.


## Actor netting / claims


Comentario profesional:

- Este cuadro sirve para conversaciones de gobernanza familiar.
- Responde quién puso, quién pagó, quién recibió repagos, quién quedó debiendo y a quién.
- Debe leerse por moneda y con la convención explícita `debtor → creditor`.
- Si faltan métricas actor-level, la notebook lo muestra como `missing_metric`, indicando una mejora de backend.

In [11]:
actor_netting_specs = [
    {
        "statement": "Actor netting / claims",
        "section": "Aportes",
        "line": "Funding por actor",
        "metric_id": "FUND.CONTRIB.BY_ACTOR",
        "dimension_policy": "expand",
        "order": 100,
        "comment": "Aportes por actor. No son ingresos operativos."
    },
    {
        "statement": "Actor netting / claims",
        "section": "Deuda / claims",
        "line": "Claims abiertos por contraparte",
        "metric_id": "ID.DEBT.OPEN.BY_COUNTERPARTY",
        "dimension_policy": "expand",
        "order": 200,
        "comment": "Stock abierto por deudor-acreedor. Usar para ver quién debe a quién."
    },
    {
        "statement": "Actor netting / claims",
        "section": "Actividad de deuda",
        "line": "Nuevos claims por contraparte",
        "metric_id": "ID.DEBT.ACTIVITY.NEW_CLAIMS",
        "dimension_policy": "expand",
        "order": 300,
        "comment": "Nuevas obligaciones creadas durante el año por actor/contraparte si está disponible."
    },
    {
        "statement": "Actor netting / claims",
        "section": "Actividad de deuda",
        "line": "Repagos por contraparte",
        "metric_id": "ID.DEBT.ACTIVITY.REPAYMENTS",
        "dimension_policy": "expand",
        "order": 310,
        "comment": "Repagos del año por actor/contraparte si está disponible."
    }
]

actor_netting_table = build_statement_table(metrics, actor_netting_specs)
display_statement(actor_netting_table, "Actor netting / claims", "Vista gerencial para entender posiciones y movimientos por actor.")

## Actor netting / claims

Vista gerencial para entender posiciones y movimientos por actor.

section,line,Currency,2022,2023,2024,2025,2026,value_status,professional_comment,caveat
Actividad de deuda,Nuevos claims por contraparte — debtor_creditor: Alejandro -> MI,USD,s/d,s/d,180,0,0,available,,Debt movement; not OPEX or funding unless classified elsewhere. | Debt movement; not OPEX or funding unless classified elsewhere. | Debt movement; not OPEX or funding unless classified elsewhere.
Actividad de deuda,Nuevos claims por contraparte — debtor_creditor: Alejandro -> PM,USD,s/d,6.703,0,0,0,available,,Debt movement; not OPEX or funding unless classified elsewhere. | Debt movement; not OPEX or funding unless classified elsewhere. | Debt movement; not OPEX or funding unless classified elsewhere.
Actividad de deuda,Nuevos claims por contraparte — debtor_creditor: Hector -> MI,USD,s/d,s/d,s/d,490,0,available,,Debt movement; not OPEX or funding unless classified elsewhere. | Debt movement; not OPEX or funding unless classified elsewhere.
Actividad de deuda,Nuevos claims por contraparte — debtor_creditor: PM -> MI,USD,s/d,5.806,2.750,268,0,available,,Debt movement; not OPEX or funding unless classified elsewhere. | Debt movement; not OPEX or funding unless classified elsewhere. | Debt movement; not OPEX or funding unless classified elsewhere.
Actividad de deuda,Nuevos claims por contraparte — debtor_creditor: PM -> Primos,USD,s/d,897,0,0,0,available,,Debt movement; not OPEX or funding unless classified elsewhere. | Debt movement; not OPEX or funding unless classified elsewhere. | Debt movement; not OPEX or funding unless classified elsewhere.
Actividad de deuda,Repagos por contraparte — debtor_creditor: Alejandro -> MI,USD,s/d,s/d,0,0,0,available,,Debt movement; not OPEX or funding unless classified elsewhere. | Debt movement; not OPEX or funding unless classified elsewhere. | Debt movement; not OPEX or funding unless classified elsewhere.
Actividad de deuda,Repagos por contraparte — debtor_creditor: Alejandro -> PM,USD,s/d,0,0,0,372,available,,Debt movement; not OPEX or funding unless classified elsewhere. | Debt movement; not OPEX or funding unless classified elsewhere. | Debt movement; not OPEX or funding unless classified elsewhere.
Actividad de deuda,Repagos por contraparte — debtor_creditor: Hector -> MI,USD,s/d,s/d,s/d,0,0,available,,Debt movement; not OPEX or funding unless classified elsewhere. | Debt movement; not OPEX or funding unless classified elsewhere.
Actividad de deuda,Repagos por contraparte — debtor_creditor: PM -> MI,USD,s/d,207,90,2.205,4.494,available,,Debt movement; not OPEX or funding unless classified elsewhere. | Debt movement; not OPEX or funding unless classified elsewhere. | Debt movement; not OPEX or funding unless classified elsewhere.
Actividad de deuda,Repagos por contraparte — debtor_creditor: PM -> Primos,USD,s/d,0,0,0,0,available,,Debt movement; not OPEX or funding unless classified elsewhere. | Debt movement; not OPEX or funding unless classified elsewhere. | Debt movement; not OPEX or funding unless classified elsewhere.


## Calidad de datos y salvedades


Comentario profesional:

- La calidad de datos es parte del estado financiero gerencial.
- Unknowns, leakage, cash safety y debt reconciliation deben estar visibles.
- Una celda `s/d` o `unavailable` es preferible a mostrar un número inseguro.

In [12]:
data_quality_specs = [
    {
        "statement": "Calidad de datos y salvedades",
        "section": "Clasificación",
        "line": "Classification coverage",
        "metric_id": "DQ.CLASSIFICATION.COVERAGE",
        "dimension_policy": "total_only",
        "order": 100,
        "comment": "Cobertura de clasificación semántica. Si baja, revisar unknown/review-required.",
        "format_hint": "percent"
    },
    {
        "statement": "Calidad de datos y salvedades",
        "section": "Clasificación",
        "line": "Unknown / ambiguous amount",
        "metric_id": "DQ.UNKNOWN.AMOUNT",
        "dimension_policy": "total_only",
        "order": 110,
        "comment": "Monto anual no clasificado o ambiguo. Debe auditarse antes de cerrar lectura ejecutiva."
    },
    {
        "statement": "Calidad de datos y salvedades",
        "section": "OPEX QA",
        "line": "OPEX leakage amount",
        "metric_id": "DQ.OPEX.LEAKAGE.AMOUNT",
        "dimension_policy": "total_only",
        "order": 200,
        "comment": "Posible filtración de gasto personal/distribución/deuda dentro de OPEX. Debe tender a cero."
    },
    {
        "statement": "Calidad de datos y salvedades",
        "section": "Cash QA",
        "line": "Cash frontend-safe",
        "metric_id": "DQ.CASH.FRONTEND_SAFE",
        "dimension_policy": "total_only",
        "order": 300,
        "comment": "Indica si la caja mostrada proviene de fuente validada. Si no, caja debe ser s/d."
    },
    {
        "statement": "Calidad de datos y salvedades",
        "section": "Debt QA",
        "line": "Debt activity reconciliation",
        "metric_id": "DQ.DEBT.ACTIVITY.RECONCILIATION",
        "dimension_policy": "total_only",
        "order": 400,
        "comment": "Estado de reconciliación entre actividad de deuda y posición final."
    }
]

data_quality_table = build_statement_table(metrics, data_quality_specs)
display_statement(data_quality_table, "Calidad de datos y salvedades", "Checks y métricas que condicionan la confianza en los cuadros anteriores.")

## Calidad de datos y salvedades

Checks y métricas que condicionan la confianza en los cuadros anteriores.

section,line,Currency,2022,2023,2024,2025,2026,value_status,professional_comment,caveat
Cash QA,Cash frontend-safe,N/A,0,0,0,0,0,available,,
Clasificación,Classification coverage,ARS,"100,0%","100,0%","100,0%","100,0%","100,0%",available,,
Clasificación,Classification coverage,USD,s/d,"100,0%","100,0%","100,0%","100,0%",available,,
Clasificación,Unknown / ambiguous amount,ARS,0,0,0,0,0,available,,
Clasificación,Unknown / ambiguous amount,USD,s/d,0,0,0,0,available,,
Debt QA,Debt activity reconciliation,N/A,s/d,161,315,399,203,available,,
OPEX QA,OPEX leakage amount,N/A,s/d,s/d,s/d,s/d,s/d,unavailable,,canonical source missing or required dimension unavailable


## Appendix — inventario de métricas disponibles

Este bloque es útil cuando el backend evoluciona. Permite ver qué `metric_id`, dimensiones y monedas están realmente disponibles.  
Si una línea profesional deseada aparece como `missing_metric`, revisar este inventario antes de tocar la notebook.

In [13]:
display(Markdown("### Metric inventory"))
inv_cols = ["metric_id", "dashboard_section", "dimension_name", "dimension_value", "Currency", "period", "value_status", "source_table"]
inv_cols = [c for c in inv_cols if c in metrics.columns]
inventory = metrics[inv_cols].drop_duplicates().sort_values([c for c in ["metric_id", "dimension_name", "dimension_value", "Currency", "period"] if c in inv_cols])
display(inventory)

display(Markdown("### Metric IDs"))
metric_ids = pd.DataFrame({"metric_id": sorted(metrics["metric_id"].dropna().astype(str).unique())})
display(metric_ids)

### Metric inventory

,metric_id,dashboard_section,dimension_name,dimension_value,Currency,period,value_status,source_table
214,BS.CASH.CLOSE.BOX,3. Cash and liquidity,NaN,NaN,N/A,NaN,unavailable,monthly_cash_close.csv
348,BS.CASH.FB,7. Legacy reconciliation,NaN,NaN,N/A,NaN,not_applicable,metric_contract_frontier.csv
349,BS.CASH.PM,7. Legacy reconciliation,NaN,NaN,N/A,NaN,not_applicable,metric_contract_frontier.csv
213,BS.CASH.TOTAL,3. Cash and liquidity,NaN,NaN,N/A,NaN,unavailable,monthly_cash_close.csv
54,COV.NET.AFTER_DRAWS,2. Funding and distributions,NaN,NaN,ARS,2022,available,monthly_operating_statement.csv
...,...,...,...,...,...,...,...,...
115,TR.FX.NET,treasury_fx,NaN,NaN,ARS,2026,available,monthly_operating_statement.csv
110,TR.FX.NET,treasury_fx,NaN,NaN,USD,2023,available,monthly_operating_statement.csv
112,TR.FX.NET,treasury_fx,NaN,NaN,USD,2024,available,monthly_operating_statement.csv
114,TR.FX.NET,treasury_fx,NaN,NaN,USD,2025,available,monthly_operating_statement.csv


### Metric IDs

,metric_id
0,BS.CASH.CLOSE.BOX
1,BS.CASH.FB
2,BS.CASH.PM
3,BS.CASH.TOTAL
4,COV.NET.AFTER_DRAWS
5,COV.SAVINGS_RATE
6,DIST.DIVIDENDS
7,DIST.DRAWS.BY_TYPE
8,DIST.DRAWS.PERSONAL
9,DQ.CASH.FRONTEND_SAFE


## Appendix — QA raw tables

Mostrar QA cruda permite distinguir problemas de datos, publish, métricas faltantes, cross-currency y artifact contracts.

In [14]:
if not annual_qa.empty:
    display(Markdown("### Annual dashboard QA"))
    display(annual_qa)
else:
    display(Markdown("### Annual dashboard QA: missing"))

if not publish_qa.empty:
    display(Markdown("### Publish contract QA"))
    display(publish_qa)
else:
    display(Markdown("### Publish contract QA: missing"))

if not artifact_contracts.empty:
    display(Markdown("### Artifact contracts"))
    display(artifact_contracts.head(200))
else:
    display(Markdown("### Artifact contracts: missing"))

### Annual dashboard QA

,check,status,detail,severity
0,annual_metrics_use_only_canonical_sources,pass,"sources=['metric_contract_frontier.csv', 'mont...",error
1,no_raw_stage_d_sources,pass,"sources=['metric_contract_frontier.csv', 'mont...",error
2,no_legacy_views_as_canonical,pass,"sources=['metric_contract_frontier.csv', 'mont...",error
3,annual_flows_sum_monthly_flows,pass,flow rules documented,error
4,annual_stocks_use_last_close,fail,stock rules documented,error
5,ratios_use_annual_aggregates,pass,ratio rules documented,error
6,no_cross_currency_aggregation,pass,money metrics carry Currency,error
7,cash_unavailable_without_frontend_safe_rows,pass,"cash_statuses=['not_applicable', 'unavailable']",error
8,debt_stock_not_mixed_with_flows,pass,debt OPEN/POSITION are stock,error
9,debt_activity_reconciles_or_residual_visible,pass,adjustments and reconciliation metric present,error


### Publish contract QA

,check,status,detail,severity
0,publish_bundle_labels_all_artifacts,pass,"classes=['artifact_contracts.csv', 'canonical_...",error
1,no_unsafe_artifacts_in_public_contract,pass,raw debt files are diagnostic/internal only,error
2,legacy_artifacts_labeled_legacy,pass,legacy annual views under legacy_reconciliation,error
3,debt_stock_activity_separated,fail,debt stock/activity contracts are properly sep...,error


### Artifact contracts

,name,relpath,artifact_role,accounting_nature,grain,currency_policy,frontend_suitability,source_authority,notes,publish_class
0,annual_balance_dashboard_metrics.csv,canonical_dashboard/annual_balance_dashboard_m...,canonical_source,mixed,annual,by_currency,row_level_or_metric_level,frontend_contract,Annual dashboard-ready metric series built onl...,public_contract
1,annual_balance_dashboard_qa.csv,canonical_dashboard/annual_balance_dashboard_q...,qa,quality,mixed,not_money,internal_only,diagnostic_only,QA artifact.,internal_diagnostic
2,frontend_metric_series.csv,canonical_dashboard/frontend_metric_series.csv,canonical_source,mixed,mixed,by_currency,row_level_or_metric_level,frontend_contract,Frontend metric series; suitability is carried...,public_contract
3,build_manifest.json,internal_diagnostic/build_manifest.json,meta,unknown,mixed,not_money,internal_only,diagnostic_only,Metadata artifact.,internal_diagnostic
4,debt_status_reconciliation.csv,internal_diagnostic/debt_status_reconciliation...,diagnostic,mixed,mixed,by_currency,internal_only,diagnostic_only,Debt engine evidence; use monthly_debt_positio...,internal_diagnostic
5,frontier_source_qa.csv,internal_diagnostic/frontier_source_qa.csv,qa,quality,mixed,not_money,internal_only,diagnostic_only,QA artifact.,internal_diagnostic
6,metrics_frontier_qa.csv,internal_diagnostic/metrics_frontier_qa.csv,qa,quality,mixed,not_money,internal_only,diagnostic_only,QA artifact.,internal_diagnostic
7,balance_cash_y.csv,legacy_reconciliation/balance_cash_y.csv,legacy,mixed,annual,by_currency,safe_with_caveat,legacy_compatibility,Legacy compatibility view; prefer metric_contr...,legacy_reconciliation
8,income_statement_y.csv,legacy_reconciliation/income_statement_y.csv,legacy,mixed,annual,by_currency,safe_with_caveat,legacy_compatibility,Legacy compatibility view; prefer metric_contr...,legacy_reconciliation
9,annual_balance_dashboard_contract.csv,public_contract/annual_balance_dashboard_contr...,canonical_source,mixed,annual,by_currency,safe_with_caveat,frontend_contract,"Contract for annual dashboard metrics, includi...",public_contract


## Export opcional

Exporta los cuadros armados por esta notebook a Excel. El Excel no es source of truth; es un output para revisar/compartir.

In [15]:
EXPORT_DIR = REPO_ROOT / "out" / "professional_pack"
EXPORT_DIR.mkdir(parents=True, exist_ok=True)
EXPORT_XLSX = EXPORT_DIR / "annual_balance_dashboard_notebook_export.xlsx"

tables_to_export = {
    "operating": operating_table,
    "funds_bridge": funds_bridge_table,
    "settlement": settlement_table,
    "position": position_table,
    "debt_rollforward": debt_rollforward_table,
    "actor_netting": actor_netting_table,
    "data_quality": data_quality_table,
    "metric_inventory": inventory,
    "readiness": readiness,
}

with pd.ExcelWriter(EXPORT_XLSX) as writer:
    for sheet_name, table in tables_to_export.items():
        table.to_excel(writer, sheet_name=sheet_name[:31], index=False)

display(Markdown(f"Export written to: `{EXPORT_XLSX}`"))

Export written to: `/home/matias/repos/accounting-backend/out/professional_pack/annual_balance_dashboard_notebook_export.xlsx`

## Próximas decisiones profesionales

Este primer diseño ya permite leer el dashboard anual, pero quedan decisiones que conviene validar con contador/profesional:

1. Cómo nombrar formalmente `COV.NET.AFTER_DRAWS`: cobertura gerencial, no resultado contable puro.
2. Si `Settlement channels` debe convertirse en métricas anuales oficiales del backend o quedar como drilldown.
3. Si `Equity bridge` requiere opening equity manual y valuación/exclusión explícita de propiedades.
4. Cómo presentar claims entre actores con convención de signo única.
5. Cómo tratar `refinanciado`, devengamientos no-cash e intereses futuros.
6. Si se necesita una vista `cash-basis` además de la vista operativa/accrual-like.